# The Pixhawk ecosystem


The Pixhawk world is the default answer to "what flies the aircraft" for anyone
not buying a DJI or a Skydio. It is worth understanding as a whole, because the
name covers four different things that are routinely conflated:

- an **open hardware standard** — a set of published documents describing
  connectors, board interfaces and reference designs;
- a **board family** — the FMUv2 through FMUv6X-RT generations built to those
  documents by a dozen manufacturers;
- a **trademark** — use of the Pixhawk name is conditional on compliance;
- loosely, the **firmware ecosystem** around it, which is really two
  independent projects (PX4 and ArduPilot) that happen to run on the same
  boards.

Core take: **Pixhawk standardizes the flight controller, and deliberately
nothing above it.** It gives a stabilized, sensor-redundant, failsafe-managed
aircraft with a well-specified command interface, and stops there. Everything
that makes an aircraft autonomous in the interesting sense — state estimation
from cameras, mapping, deciding where to fly — lives on a companion computer
and is not standardized at all.

Terms: FMU = flight management unit, the autopilot board proper.

## Who runs it

The standards are produced by the **Pixhawk Special Interest Group**, convened
by the [Dronecode Foundation][dronecode], which is a Linux Foundation
collaborative project. Dronecode is also the legal home of PX4, MAVLink,
QGroundControl and MAVSDK, and holds those trademarks.

Two governance facts shape the ecosystem more than any technical decision:

- **ArduPilot is not part of it.** ArduPilot separated from Dronecode and is run
  by its own board, under GPLv3, while PX4 is BSD 3-clause under Dronecode. The
  same board therefore has two firmware options with different licences,
  different release cadences and different feature sets — and no shared
  roadmap.
- **Contribution is concentrated.** Dronecode's own 2025 review puts Auterion at
  roughly 25% of PX4 contributions, the largest single organization, with
  independent contributors at ~28%. Auterion also sells a closed commercial
  distribution built on PX4. That is a normal open-core arrangement, but it does
  mean the largest voice in the reference implementation is also a vendor
  competing in the market the standard serves.

The practical read: the hardware standard is genuinely open and multi-vendor,
the firmware is open but strongly steered, and the two firmwares that run on
the hardware answer to different masters.

[dronecode]: https://dronecode.org/

## The standards themselves

The [Pixhawk Standards repository][std] is the authoritative list. Documents are
numbered DS-0xx; DS-001 to DS-008 are reserved and unused.

| Document | Covers | Status |
|---|---|---|
| DS-009 | Connector standard | active |
| DS-010 | Pixhawk Autopilot Bus (PAB) | active |
| DS-011 | Autopilot v5X | active |
| DS-012 | Autopilot v6X | active |
| DS-013 | Smart battery | **deprecated** |
| DS-014 | Payload bus | active |
| DS-015 | (superseded by the CAN drone SIG) | **deprecated** |
| DS-016 | Autopilot v6U | **draft** |
| DS-017 | Radio interface | active |
| DS-018 | Autopilot v6C | active |
| DS-019 | Versions and revisions | active |
| DS-020 | Autopilot v6X-RT | active |

[std]: https://github.com/pixhawk/Pixhawk-Standards

Three of these carry nearly all the practical weight:

- **DS-009** fixes JST-GH connector pinouts. This is why a GNSS module, a
  telemetry radio or an airspeed sensor from one vendor plugs into another
  vendor's board without an adapter. Unglamorous and the single most valuable
  document in the set.
- **DS-010, the Pixhawk Autopilot Bus**, splits the autopilot into an **FMU
  module** (microcontroller, inertial measurement units, barometer,
  magnetometer) and a **carrier board** (connectors, power, motor outputs,
  whatever else the integrator wants). A PAB-compliant FMU drops into any
  PAB-compliant carrier.
- **DS-012 and friends** prescribe the sensor complement per generation, which
  is what lets PX4 and ArduPilot ship a single firmware target that works across
  every vendor's implementation of that generation.

DS-010 is the one with strategic consequences: it means a custom vehicle can
have a fully custom carrier board — the exact connectors, power tree and form
factor wanted — while the certified, sensor-calibrated, firmware-supported part
is bought off the shelf. It is the reason the Jetson carrier market converged on
a PAB socket rather than each vendor inventing an autopilot.

## Generations

Board generations are named FMUv*N*. The number tracks the reference design, not
the vendor.

| Generation | Processor | Notes |
|---|---|---|
| FMUv2 / v3 | STM32F4 | 2013-2016 era, 1 MB flash limit, legacy |
| FMUv4 | STM32F4 | Pixracer class, small and light |
| FMUv5 | STM32F7 | the volume workhorse for years |
| FMUv5X | STM32F7 | first PAB split of FMU and carrier |
| FMUv6C | STM32H7 | cost-reduced, no Ethernet |
| FMUv6X | STM32H7 | the current mainstream: triple redundant IMU, Ethernet |
| FMUv6X-RT | NXP i.MX RT1176 | same standard, non-ST processor |
| FMUv6U | - | **draft**, not shipping |

Two things worth extracting from that table:

- **FMUv6X is the current default.** Triple redundant, vibration-isolated
  inertial measurement units, separate power domains, Ethernet, and support in
  both firmwares. If a project does not have a specific reason to choose
  otherwise, this is the choice.
- **FMUv6X-RT proves the standard works.** NXP built the same interface around a
  completely different processor family, and firmware support followed. A
  standard that survives a processor-vendor change is a real standard.

No FMUv7 has been announced. The visible activity is the v6U draft and refinement
inside the v6 family, which suggests the generation has several years left.

Note the naming trap: **"Pixhawk 6X" is a Holybro product, "FMUv6X" is the
standard it implements.** CUAV, ARK, RaccoonLab and NXP all build FMUv6X-class
boards under their own names. PX4's documentation splits boards into *Pixhawk
standard* (fully compliant, trademark-licensed, tested by the project) and
*manufacturer supported* (maintained by the vendor), and the distinction is
worth checking before committing to a board.

## The hardware landscape

Prices are July 2026 (see [notebook 08](08_flight_compute_landscape.ipynb) for
the wider compute comparison and the caveats).

| Board | Generation | Vendor | Price | Note |
|---|---|---|---|---|
| [Pixhawk 6X][h6x] | FMUv6X | Holybro | $166.99 | the volume default |
| Pixhawk 6X Pro | FMUv6X | Holybro | - | higher-spec variant |
| [Pixhawk 6C mini][h6c] | FMUv6C | Holybro | from $130.99 | no Ethernet |
| [ARKV6X][arkv6x] | FMUv6X | ARK | $400.00 | 5 g, US-built |
| Pixhawk V6X | FMUv6X | CUAV | - | |
| FMUv6X | FMUv6X | RaccoonLab | - | CAN-focused vendor |
| [MR-VMU-RT1176][nxp] | FMUv6X-RT | NXP | $392.70 | 13-week lead |
| [Cube Orange+][cube] | CubePilot | Hex/ProfiCNC | $277 | **not PAB** |
| [Pixracer Pro][mro] | FMUv5-class | mRo | $349.90 | 8.9 g |

[h6x]: https://holybro.com/products/pixhawk-6x
[h6c]: https://holybro.com/products/pixhawk-6c-mini
[arkv6x]: https://arkelectron.com/product/arkv6x/
[nxp]: https://www.digikey.com/en/products/detail/nxp-usa-inc/MR-VMU-RT1176/26220861
[cube]: https://irlock.com/products/cube-orange-plus-standard-set
[mro]: https://store.3dr.com/pixracer-pro/

The price spread on functionally equivalent hardware — $167 to $400 for the same
generation — is assembly location and supply-chain provenance rather than
capability.

**CubePilot is the notable defector.** The Cube line is widely deployed and well
regarded, is the reference platform for much of the ArduPilot world, and uses
its own carrier interface rather than PAB. Choosing a Cube means choosing the
Cube carrier ecosystem too.

## Buses and peripherals

What actually connects to an autopilot, and which of it is standardized:

| Interface | Used for | Standardization |
|---|---|---|
| JST-GH serial / I2C | GNSS, telemetry radio, airspeed, rangefinder | DS-009, solid |
| CAN (DroneCAN) | ESCs, GNSS, power, servos — bus topology, less wiring | de facto |
| PWM / DShot | direct motor drive | universal, DShot now bidirectional |
| RC input | SBUS, CRSF, ELRS receivers | de facto |
| Ethernet | companion computer link on v6X-class | DS-012 |
| USB | configuration, ground station, some companion links | universal |
| Payload bus | gimbals, cameras, sensors | DS-014, thinly adopted |

Two of these deserve comment.

**CAN is a fork story.** The original UAVCAN split in 2022: v0 was renamed
**DroneCAN** and v1 became **Cyphal**. Both PX4 and ArduPilot use DroneCAN as
the practical peripheral bus; Cyphal is the technically better-designed
successor, with support that is real but not the default path. Anyone buying CAN
peripherals should assume DroneCAN unless proven otherwise, and note that the
newer protocol has been "the future" for four years.

**Bidirectional DShot arrived in PX4 v1.16**, sponsored by ARK. It reads motor
RPM back over the same wire used to command throttle. This is a genuinely useful
capability for a project that cares about vibration, propeller state or motor
health, and it removes a reason to run separate telemetry-capable ESCs.

## The two firmwares

The same board runs either. They are not interchangeable in practice.

| | PX4 | ArduPilot |
|---|---|---|
| Licence | BSD 3-clause | GPLv3 |
| Governance | Dronecode / Linux Foundation | independent board |
| Latest stable | v1.17.0, May 2026 | 4.7.0, July 2026 |
| Architecture | modular, uORB message bus, NuttX | monolithic, HAL abstraction |
| ROS 2 story | native — uXRCE-DDS, `px4_ros2` library | MAVLink bridge, DDS support newer |
| Built-in path planning | no | yes — BendyRuler, Dijkstra |
| Typical culture | research, OEM products, ROS | hobby-to-commercial, deep configurability |

The licence difference is the one that reaches a product decision: BSD lets a
company ship modified PX4 without publishing changes, GPLv3 does not. That
single fact explains most of the commercial gravitation toward PX4 and most of
the feature depth in ArduPilot.

For indoor autonomy specifically the interesting asymmetries are:

- **ArduPilot has object avoidance in the flight controller.** BendyRuler is a
  local planner, Dijkstra plans around known fences, and the two compose. It is
  crude compared to a proper mapping planner, but it exists, ships, and runs
  without a companion computer.
- **PX4 has the better companion-computer interface.** Collision prevention
  stops the vehicle before an obstacle but does not route around it; the routing
  is expected to come from outside.

Which is to say the two projects drew the autonomy boundary in different places.
PX4 assumes a companion computer will do the thinking; ArduPilot assumes it may
not be there.

## Talking to a Pixhawk from a computer

Three generations of interface coexist, and picking the right one matters more
than the board choice.

**MAVLink** is the lingua franca — a compact message protocol over serial, UDP
or TCP, spoken by every ground station and both firmwares. `MAVSDK` (C++,
Python) and `MAVROS` (ROS 1/2 bridge) wrap it. It is universal, well documented,
and shows its age: request-response semantics, limited rates, and a message set
that grew by accretion.

**uXRCE-DDS** is the modern PX4 path. Micro XRCE-DDS ("eXtremely Resource
Constrained Environment" DDS) runs a thin client on the flight controller
talking to an agent on the companion computer, which bridges into the data
distribution service middleware ROS 2 is built on. The effect is that PX4's
internal uORB topics appear directly as ROS 2 topics — `/fmu/out/*` and
`/fmu/in/*` — with no translation layer. It replaced the older micro-RTPS bridge
in v1.14.

**The [PX4 ROS 2 Interface Library][lib]** is the newest layer and the most
consequential for autonomy work. It provides:

- a **control interface** for writing flight modes *in ROS 2* that register
  themselves dynamically with PX4, appear to a ground station as native modes,
  and fall back to the built-in mode if the external one fails;
- **mode executors** — a state machine that can sequence modes, e.g. takeoff,
  then a custom capture mode, then return-to-launch;
- a **navigation interface** for feeding external position estimates, i.e. the
  supported way to plug a visual-inertial odometry system into the estimator;
- setpoint types from smooth position-and-heading goto down to direct actuator
  commands.

[lib]: https://docs.px4.io/main/en/ros2/px4_ros2_interface_lib

This is the correct attachment point for an exploration or capture policy, and
it is a much better one than the traditional approach of streaming offboard
setpoints and hoping the failsafe logic agrees. Minimum PX4 v1.15, with real
improvements in v1.17; parts are still marked experimental, which is worth
weighting against the alternative of the older offboard mode.

## Indoor and GPS-denied flight

The estimator side is solved and unglamorous. PX4's **EKF2** accepts external
vision pose and odometry, and will fuse it as the primary position source with
no satellite fix; ArduPilot has the equivalent through `VISION_POSITION_ESTIMATE`
and its EKF3. Both are well-trodden: motion-capture flight in a lab is a
standard demo, and a visual-inertial odometry source is the same interface with
a different producer.

What has to come from outside, in both firmwares:

- **the odometry itself** — no Pixhawk-class board has the compute or the camera
  interfaces to run visual-inertial odometry, so this is a companion-computer
  job;
- **the map** — no volumetric or occupancy representation exists in either
  firmware;
- **the goal** — nothing decides where to fly.

The gap between "the interfaces exist" and "there is a working stack" is real,
and it is visible in the state of the reference material:

- **PX4-Avoidance is archived.** The project's own obstacle-avoidance companion
  stack is no longer maintained.
- **The PX4 Vision Autonomy Development Kit is discontinued.** The kit the
  documentation still recommends — a quadcopter with a Pixhawk 6C, an UP Core
  companion computer and a Structure Core depth camera, $1,889.99 for the v1.5 —
  is not orderable, and the v1 is sold out.
- **The recommended depth sensors keep disappearing.** PX4's own vision docs
  cited the Intel RealSense T265, which was discontinued; the surviving RealSense
  depth cameras are intermittently out of stock.

So the honest position is that Pixhawk offers a **well-specified socket for
indoor autonomy with nothing plugged into it**. That is not a criticism of the
standard — it is exactly the boundary the project drew — but it does mean an
indoor autonomous vehicle built on Pixhawk is an integration project on the
companion side, with the flight-control half already solved.

## Strengths and weaknesses, plainly

Strengths:

- **Genuine multi-vendor hardware.** A board can be replaced with another
  vendor's without redesigning the harness or the firmware target. Very little
  else in robotics offers this.
- **Two independent firmware implementations** of the same interface, which is
  the strongest possible evidence that the standard is real.
- **Failsafe maturity.** Battery, RC loss, geofence, estimator divergence,
  landing detection — a decade of accumulated flight-safety logic that would be
  extremely expensive to reproduce and dangerous to skip.
- **Cheap.** A capable current-generation flight controller is $130-400, an order
  of magnitude under integrated compute-plus-autopilot boards.
- **The interface to a companion computer is now good**, which was not true
  three years ago.

Weaknesses:

- **The standard covers the autopilot only.** Everything above it — perception,
  mapping, planning — is unstandardized, and the reference implementations of
  that layer are archived or discontinued.
- **Compliance is a spectrum.** Vendors advertise "Pixhawk compatible" with
  varying fidelity; PX4's *standard* versus *manufacturer supported* distinction
  exists precisely because of this.
- **Two firmwares means two ecosystems.** Documentation, parameter names, tuning
  knowledge and community answers do not transfer, and choosing wrongly early is
  expensive.
- **Peripheral protocol drift.** DroneCAN versus Cyphal has been unresolved for
  four years.
- **Integration labour is the real cost.** The board is cheap; the airframe,
  power, sensor calibration, tuning and safety validation are not.

## Relevance to indoor autonomous capture

Reading the ecosystem against a project that wants a drone to fly itself around
a building and take usable photographs:

- **The flight-control problem is bought, not built.** Stabilization, motor
  control, failsafes, battery management, external-position fusion and a
  documented command interface all come free. Attempting any of it directly is a
  waste of effort.
- **The natural architecture is a split stack**: a Pixhawk-class flight
  controller plus a Linux companion computer over Ethernet, uXRCE-DDS between
  them, ROS 2 on top. This buys a current userland, current tooling, and a
  flight controller replaceable in a week — against the alternative of an
  integrated board like the VOXL, which trades that flexibility for ~150 g and
  a pre-integrated autonomy stack.
- **The `px4_ros2` control interface is where a capture policy attaches**, as a
  registered custom mode with a mode executor sequencing takeoff, exploration and
  return. This is a materially better contract than streaming offboard setpoints,
  because the failsafe system understands it.
- **The navigation interface is where odometry attaches**, whether that comes
  from a real visual-inertial system or from a simulator. This matters for
  simulation work: PX4 does not distinguish, so the same planner code drives
  software-in-the-loop, hardware-in-the-loop and a real vehicle.
- **Nothing in the ecosystem competes with an exploration policy.** ArduPilot's
  BendyRuler is a local avoider, PX4's collision prevention is a brake. Neither
  chooses viewpoints, tracks coverage or scores image quality.

Where it argues against Pixhawk: a Pixhawk plus a companion computer is a
~280-340 g, several-box assembly against an 11 g integrated board that already
ships odometry, mapping and avoidance. For a bench and a research schedule the
split stack wins on iteration speed; for a small airframe or a product it does
not.

## References

Standards and governance:

- [Pixhawk Standards repository](https://github.com/pixhawk/Pixhawk-Standards)
- [Pixhawk project site](https://pixhawk.org/standards/)
- [Dronecode Foundation](https://dronecode.org/)

Hardware:

- [PX4: Pixhawk standard autopilots](https://docs.px4.io/main/en/flight_controller/autopilot_pixhawk_standard)
- [PX4: reference flight controller design](https://docs.px4.io/main/en/hardware/reference_design.html)

Firmware and interfaces:

- [PX4 v1.17 release notes](https://docs.px4.io/main/en/releases/1.17)
- [PX4 v1.16 release notes](https://docs.px4.io/main/en/releases/1.16)
- [ArduPilot releases](https://github.com/ArduPilot/ardupilot/releases)
- [PX4 ROS 2 Interface Library](https://docs.px4.io/main/en/ros2/px4_ros2_interface_lib)
- [PX4 ROS 2 Control Interface](https://docs.px4.io/main/en/ros2/px4_ros2_control_interface)
- [PX4 CAN: DroneCAN and Cyphal](https://docs.px4.io/main/en/can/)
- [DroneCAN](https://dronecan.github.io/)

Autonomy:

- [PX4 computer vision](https://docs.px4.io/main/en/computer_vision/)
- [ArduPilot object avoidance](https://ardupilot.org/copter/docs/common-object-avoidance-landing-page.html)
- [ArduPilot Dijkstra with BendyRuler](https://ardupilot.org/copter/docs/common-oa-dijkstrabendyruler.html)
- [PX4 Vision Autonomy Development Kit](https://docs.px4.io/main/en/complete_vehicles_mc/px4_vision_kit)

Figures and availability date from 2026-08-05.